<a href="https://colab.research.google.com/github/mariobet23440/Proyecto-3---M-todos-Num-ricos-1/blob/main/Proyecto3MetodosNumericos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***MOTOR DC - GEMELO DIGITAL Y CONTROL PID***

```Proyecto 3 - Métodos Numéricos 1```

***Creado por:***
*   Mario Betancourt (23440)
*   Pongan sus nombres y (carnets)

## ***HISTORIAL DE REVISIONES***
*   21/11/2026 - Creación de archivo y método de Euler
*   Elemento de lista



# ***INTRODUCCIÓN***
Le damos la bienvenida al cuaderno de Jupyter del Proyecto 3 del curso Métodos Numéricos 1 de la Universidad del Valle de Guatemala. En este proyecto trabajaremos en la aplicación de métodos numéricos para calcular derivadas e integrales y métodos de resolución de ecuaciones diferenciales al análisis de la dinámica de un motor DC.

De acuerdo con la guía, el motor DC se modela a través de un sistema de tres ecuaciones diferenciales acopladas:

$$\begin{cases}
L\frac{di}{dt} = V-Ri-K_e \omega \\
J\frac{d\omega}{dt} = K_t i - b\omega -\tau_c \text{ sign}(\omega) - \tau_{carga} \\
\frac{d\theta}{dt} = \omega
\end{cases}$$

Donde los símbolos representan las siguientes cantidades físicas

*   $L$ - Inductancia del motor [Henries, H]
*   $V$ - Voltaje aplicado en las terminales del motor [Volts, V]
*   $R$ - Resistencia del motor [Ohms, $\Omega$]
*   $K_e$ - Constante de fuerza contraelectromotriz [$V/(rad/s)$]
*   $\omega$ - Velocidad angular del motor [radianes por segundo, $rad/s$]
*   $J$ - Inercia del eje del motor [$kg * m^2$]
*   $K_t$ - Constante de torque del motor [$(N*m)/A$]
*   $b$ - Coeficiente de amortiguamiento viscoso [$(N*m)/(rad/s)$]
*   $\tau_c$ - Torque de fricción de Coulomb [$N*m$]
*   $\tau_{carga}$ - Torque de carga [$N*m]

Un estudio de estos modelos en cursos de sistemas de control sugiere que el sistema de ecuaciones representa a un motor DC controlado por armadura. Este modelo de motores DC supone que la corriente en el estator (corriente de campo) es constante y el control de movimiento se realiza por la corriente del devanado del rotor (devanado de armadura). Este modelo de motores es aplicable también a motores DC de imanes permanentes. El motor DC JGA25-370 entra dentro de esta categoría. Para más información, vease Nise

(AGREGAR REFERENCIA POR FAVOR, capítulo 4 no recuerdo donde estaba)


A diferencia del modelo lineal presentado en Nise (no sé, no sé, no sé) el modelo analizado en este proyecto agrega una dificultad adicional: *se considera un término no lineal para la fricción de Coulomb*. Aunque es posible linealizar el sistema y analizarlo a través de Transformadas de Laplace (como se hace en Sistemas de Control 1), ese no es el objetivo de este proyecto. Al fin y al cabo, este es un curso de métodos numéricos y no un curso de cómo aproximar cosas. Por lo tanto, a diferencia de lo que se haría en un análisis de sistemas lineales, en este proyecto resolveremos la ecuación original con todas sus no linealidades.

# ***PARTE 1 - SOLUCIÓN DE SISTEMAS DE ECUACIONES DIFERENCIALES***

Ahora que fijamos como objetivo resolver el sistema de ecuaciones diferenciales mostrado anteriormente, surge la importante pregunta de cómo debemos proceder. Empecemos tratando de entender cómo resolver un sistema de ecuaciones a través del método de Euler.

## **1.1. El Método de Euler para una EDO de primer orden**
El método de Euler es fácil y muy intuitivo de definir para un Problema de Valores Iniciales (PVI) con una EDO de primer orden:

$$\begin{cases}
\frac{dy}{dx}=f(x,y)\\
y(x_0) = y_0
\end{cases}
$$

Donde $x_0$ y $y_0$ son constantes arbitrarias. Para aproximar la solución a este PVI, utilizamos diferencias finitas hacia adelante para aproximar la derivada en el rango $x \in [x_k, x_{k+1}]$ donde $x_{k+1} = x_k + h$:

$$\frac{dy}{dx} \approx \frac{y_{k+1} - y_k}{h} $$

Donde $y_{k+1} = y(x_{k+1})$ y $y_{k} = y(x_{k})$ son los valores de la variable independiente evaluados en los límites del rango antes definido.

Al sustituir en la EDO:
$$\frac{y_{k+1} - y_k}{h}\approx f(x,y)$$

Resuelva para $y_{k+1}$ y obtendrá la ecuación del método de Euler explícito (el método de Euler que aproxima la solución extrapolando hacia adelante usando la derivada en el punto actual):

$$y_{k+1} = y_k + hf(x_k, y_k)$$

¿Ve lo simple que es llegar a este famoso método?



## **1.2. El Método de Euler para un sistema de EDOs de primer orden**
Ahora extrapolemos esta deducción hacia un sistema de varias EDOs de variables dependientes en el vector $\textbf{y}=[y_1, y_2, ..., y_N]^T$. ¿Qué tanto podría cambiar?

Suponga que todas las ecuaciones del sistema pueden escribirse en forma canónica (aunque el lado derecho sea una no linealidad espeluznante):

$$
\begin{cases}
y_1' = f_1(x,y_1,y_2, ..., y_N)\\
y_2' = f_2(x,y_1,y_2, ..., y_N)\\
y_3' = f_3(x,y_1,y_2, ..., y_N)\\
\vdots
y_N' = f_N(x,y_1,y_2, ..., y_N)
\end{cases}
$$

Aunque no lo crea, este sistema de ecuaciones grita que puede ser expresado a través de una matriz. Quizás a simple vista no lo vea, pero tal vez al recordar cómo se define el Jacobiano le de una pista:

Sea $\mathbf{f}: \mathbb{R}^n \to \mathbb{R}^m$ una función vectorial diferenciable definida por $\mathbf{f}(\mathbf{x}) = (f_1(\mathbf{x}), f_2(\mathbf{x}), \dots, f_m(\mathbf{x}))$, donde $\mathbf{x} = (x_1, x_2, \dots, x_n)$. La *matriz jacobiana* de $\mathbf{f}$ en $\mathbf{x}$, denotada por $\mathbf{J}_{\mathbf{f}}(\mathbf{x})$ o $\frac{\partial \mathbf{f}}{\partial \mathbf{x}}$, es la matriz de $m \times n$ de sus primeras derivadas parciales:

$$
\mathbf{J}_{\mathbf{f}}(\mathbf{x}) = \frac{\partial (f_1, \dots, f_m)}{\partial (x_1, \dots, x_n)} =
\begin{pmatrix}
\frac{\partial f_1}{\partial x_1} & \frac{\partial f_1}{\partial x_2} & \cdots & \frac{\partial f_1}{\partial x_n} \\
\frac{\partial f_2}{\partial x_1} & \frac{\partial f_2}{\partial x_2} & \cdots & \frac{\partial f_2}{\partial x_n} \\
\vdots & \vdots & \ddots & \vdots \\
\frac{\partial f_m}{\partial x_1} & \frac{\partial f_m}{\partial x_2} & \cdots & \frac{\partial f_m}{\partial x_n}
\end{pmatrix}
$$

Dado que en este caso el sistema de ecuaciones diferenciales consiste de varias funciones $\textbf{y}$ dependientes de la misma variable independiente $t$, se deduce que el Jacobiano de $\textbf{y}$ (el conjunto de funciones para las que se resuelve el sistema expresados como una función vectorial) es:

$$J\textbf{y} = \left[\frac{dy_1}{dt},\frac{dy_2}{dt}, ..., \frac{dy_N}{dt}\right]^T$$

Por lo tanto, si recuerda que la derivar una función vectorial equivale a derivar cada una de sus componentes:

$$\textbf{y}' = \left[\frac{dy_1}{dt},\frac{dy_2}{dt}, ..., \frac{dy_N}{dt}\right]^T$$

Resulta obvio que:

$$\textbf{y}' = J\textbf{y}$$

¡Para sistemas de EDOs de primer orden el operador de la pendiente $f(x,y)$ se reemplaza por el Jacobiano!

Aunque los Jacobianos asustan incluso a estas alturas de la carrera, resolver el sistema de ecuaciones mediante el método de Euler es realmente sencillo: aproxime la derivada de cada variable independiente en el sistema a través de diferencias finitas y resuelva para el valor de la función en el extremo derecho del intervalo sobre el que se aproxima. Básicamente, haga lo mismo que hicimos antes para deducir que $y_{k+1} = y_k + hf(x_k, y_k)$ pero *por cada función $y_i$ en el sistema de EDOs*.

De esta forma, por cada iteración $k$, $x_k = x_0 + hk$ y por cada variable independiente:

$$
\begin{align}
(y_1)_{k+1} = (y_1)_{k} + h f_1 (x_k, (y_1)_k, (y_2)_k, ..., (y_N)_k)\\
(y_2)_{k+1} = (y_2)_{k} + h f_2 (x_k, (y_1)_k, (y_2)_k, ..., (y_N)_k)\\
\vdots\\
(y_N)_{k+1} = (y_N)_{k} + h f_N (x_k, (y_1)_k, (y_2)_k, ..., (y_N)_k)\\
\end{align}
$$

Por definición, el Jacobiano en este caso es un vector con los operadores $f_1$, $f_2$, ..., $f_N$. Por lo tanto, se deduce que para un sistema de ecuaciones las fórmulas iterativas del método de Euler pueden reescribirse en forma vectorial:

$$\textbf{y}_{k+1} =\textbf{y}_{k} + h J\textbf{y}_{k}$$

Al reunir todos los operadores en un solo paréntesis:

$$\textbf{y}_{k+1} =(1 + h J) \textbf{y}_{k}$$

Esta forma de expresar la fórmula iterativa del método de Euler es extremadamente importante. Por favor, recuérdela, que será de mucha utilidad cuando tratemos de estudiar la estabilidad del método en este caso.

## **1.3. Sistemas de EDOs de orden superior**
Ahora considere que agregamos la complejidad adicional de que en nuestro sistema de ecuaciones tenemos al menos una ecuación con una derivada de orden superior de una variable independiente. Suponga, por ejemplo, que en un sistema de ecuaciones que involucra las variables $\textbf{y} = [y_1, y_2, ..., y_N]^T$ la variable $y_i$ donde $i\in \mathbb{N} | 1 \leq i \leq N$ (revisar notación de conjunto) tiene una derivada de orden $j$-esimo en una ecuación del sistema.

La solución, en este caso, es bastante simple: *reemplace la derivada de orden superior por una derivada de primer orden utilizando variables auxiliares*.

Para ilustrar mejor esta idea, olvide que tratamos con sistemas de ecuaciones y considere lo que ocurre con una sola EDO de segundo orden:

$$\frac{d^2y}{dt^2} = \frac{dy}{dt} - 1$$

En este caso, podemos convertir la ecuación de segundo orden en un *sistema* de ecuaciones de primer orden sustituyendo la derivada de segundo orden por la derivada de una variable auxiliar. Considere que $u_1 = y$ y $u_2 = y'$. Para este problema se obtiene el sistema de EDOs de primer orden:

$$\begin{cases}
\frac{du_2}{dt}= u_2 - 1\\
\frac{du_1}{dt}= u_2
\end{cases}
$$

Por lo tanto, en un sistema de ecuaciones donde aparecen múltiples EDOs de orden superior, estas EDOs se descomponen en al menos tantas EDOs de primer orden como el orden de su derivada más alta. En este ejemplo, dado que la ecuación era de segundo orden (orden 2), se necesitaron 2 ecuaciones de primer orden.

Con esto mencionado, ya tenemos la teoría suficiente para (1) aplicar el método de Euler y generalizar a otros métodos para resolver EDOs vistos en el curso y (2) Analizar la estabilidad de la solución numérica, al menos para el método de Euler.

## **1.4. Estabilidad Numérica del Método de Euler**
Suponga que la solución exacta del sistema de EDOs en $x_k$ es $\textbf{Y}(x_k)$ (observe que se usó una letra mayúscula). Esta puede calcularse de forma exacta en las cercanías de $x_k$ al utilizar series de Taylor y agregar un término de error local por los términos de orden superior:

$$\textbf{Y}(x_{k+1}) = \textbf{Y}(x_k) + h\textbf{f}(x_k, \textbf{Y}(x_k)) + \frac{h^2}{2}\textbf{Y}''(\epsilon)$$

El error en el paso $k+1$ del método de Euler $\textbf{e}_{k+1}$ es:

$$\textbf{e}_{k+1} = \textbf{Y}(x_{k+1}) - \textbf{y}_{k+1} = [\textbf{Y}(x_k)-\textbf{y}_{k+1}] + h[\textbf{f}(x_k, \textbf{Y}(x_k)) - \textbf{f}(x_k, \textbf{y}_k)] + \frac{h^2}{2}\textbf{Y}''(\epsilon)$$

Observe que $[\textbf{f}(x_k, \textbf{Y}(x_k)) - \textbf{f}(x_k, \textbf{y}_k)] = J[\textbf{Y}(x_k)-\textbf{y}_k]$.

Definimos el error en el paso $k$ como $\textbf{e}_{k} = \textbf{Y}(x_{k}) - \textbf{y}_{k}$. Por lo tanto, el error en el paso $k+1$ se vuelve

$$\textbf{e}_{k+1} = \textbf{e}_{k} + hJ\textbf{e}_{k} + \frac{h^2}{2}\textbf{Y}''(\epsilon)$$

Si escogemos un $h$ suficientemente pequeño, el error asociado al residuo $\textbf{Y}''(\epsilon)$ debería ser despreciable comparado a los demás términos de la ecuación. Por lo tanto, de ahora en adelante, lo descartamos y aproximamos el error mediante:

$$\textbf{e}_{k+1} \approx (1 + hJ)\textbf{e}_{k}$$

Observe que el error en el paso $k+1$ es proporcional al error en el paso anterior $k$ con un **factor de amplificación** $(1+ hJ)$. Pero, por la manera en que aquí está definido, este factor es en realidad un operador.